# Loss Functions: from Squared Error to Cross-Entropy

A loss does two jobs at once, and they are easy to confuse. Its **minimum** decides what the model converges to; its **derivative** decides how fast it gets there and whether it moves at all. A loss can be perfectly correct about the first and useless about the second — which is exactly what goes wrong when squared error is used on a classifier.

$$\mathcal{L}(\theta)=\frac{1}{n}\sum_i \ell\big(y_i,\hat{y}_i(\theta)\big),
\qquad \frac{\partial\mathcal{L}}{\partial\theta}=\frac{1}{n}\sum_i\frac{\partial\ell}{\partial\hat{y}_i}\frac{\partial\hat{y}_i}{\partial\theta}$$

Six losses, in order of difficulty. For each one the notebook shows the same two pictures: the shape of $\ell$, and the shape of $\partial\ell/\partial\hat{y}$ — because everything that matters in training lives in the second.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display

plt.rcParams.update({
    "figure.dpi": 108, "font.size": 9, "axes.titlesize": 9.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3,
})

C1, C2, C3, C4, CK = "#1f77b4", "#b2182b", "#7f4fbf", "#2ca02c", "#111111"
SL = {"style": {"description_width": "108px"},
      "layout": widgets.Layout(width="330px"), "continuous_update": False}


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))


def softmax(z):
    e = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def squared(r):
    return r ** 2


def absolute(r):
    return np.abs(r)


def huber(r, delta=1.0):
    a = np.abs(r)
    return np.where(a <= delta, 0.5 * r ** 2, delta * (a - 0.5 * delta))


def d_squared(r):
    return 2 * r


def d_absolute(r):
    return np.sign(r)


def d_huber(r, delta=1.0):
    return np.clip(r, -delta, delta)


def fit_constant(y, kind, delta=1.0, grid=None):
    """Minimize a loss over a single constant prediction, by dense search."""
    g = np.linspace(y.min() - 2, y.max() + 2, 20001) if grid is None else grid
    R = g[:, None] - y[None, :]
    L = {"squared": squared, "absolute": absolute,
         "huber": lambda r: huber(r, delta)}[kind](R).mean(1)
    return g, L, float(g[int(np.argmin(L))])


print("mean vs median on 41 points with 2 large outliers:")
_y = np.r_[np.random.default_rng(0).normal(0, 1, 39), [12.0, 15.0]]
for k in ("squared", "absolute"):
    print(f"   argmin of {k:8s} loss = {fit_constant(_y, k)[2]:+.4f}")
print(f"   sample mean   = {_y.mean():+.4f}")
print(f"   sample median = {np.median(_y):+.4f}")

mean vs median on 41 points with 2 large outliers:
   argmin of squared  loss = +0.5627
   argmin of absolute loss = -0.1283
   sample mean   = +0.5631
   sample median = -0.1285


## 1 — Squared error, and why it estimates the mean

The first loss anyone meets. Its name is literal: the penalty for one point is the **area of a square** whose side is the residual, which is why one bad point can dominate a whole dataset.

$$\ell(y,\hat{y})=(y-\hat{y})^2,\qquad \frac{\partial\ell}{\partial\hat{y}}=-2(y-\hat{y})$$

Two consequences follow from that derivative being *linear in the residual*. First, setting $\sum_i(y_i-c)=0$ gives $c=\bar{y}$: **squared error is the loss whose minimiser is the mean**, and the cell above confirms the numerical argmin and the sample mean agree to the resolution of the search. Second, a point twice as far away pulls twice as hard — influence grows without bound.

The right panel is the loss as a function of the prediction, and it is a parabola. Its curvature is constant, which is what makes least squares solvable in closed form and gradient descent on it so well behaved.

In [2]:
DATA0 = np.r_[np.random.default_rng(3).normal(0.0, 1.0, 11)]


def draw_squares(c, outlier):
    y = np.r_[DATA0, outlier]
    x = np.arange(len(y))
    g, L, cmin = fit_constant(y, "squared")

    fig, ax = plt.subplots(1, 2, figsize=(11.4, 4.0))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.86, bottom=0.14, wspace=0.24)
    ax[0].axhline(c, color=C2, lw=2, label=f"prediction c = {c:.2f}")
    ax[0].axhline(y.mean(), color=C1, lw=1.4, ls="--",
                  label=f"mean = {y.mean():.2f}")
    for xi, yi in zip(x, y):
        r = yi - c
        ax[0].add_patch(mpatches.Rectangle((xi, min(yi, c)), abs(r), abs(r),
                                           facecolor=C2, alpha=0.22, edgecolor=C2))
        ax[0].plot([xi, xi], [c, yi], color="0.5", lw=0.8)
    ax[0].plot(x, y, "o", ms=6, color=CK, zorder=5)
    ax[0].set_xlim(-1, len(y) + 6); ax[0].set_ylim(min(y.min(), c) - 1.5,
                                                   max(y.max(), c) + 1.5)
    ax[0].set_aspect("equal", adjustable="box")
    ax[0].set_xlabel("sample index"); ax[0].set_ylabel("y")
    ax[0].legend(fontsize=7.5, loc="upper right")
    ax[0].set_title("each penalty is literally the area of a square")

    ax[1].plot(g, L, color=C1, lw=2)
    ax[1].axvline(c, color=C2, lw=1.6, ls="--")
    ax[1].plot([cmin], [L.min()], "*", ms=16, color=C4, zorder=5)
    ax[1].set_xlabel("prediction c"); ax[1].set_ylabel("mean squared error")
    ax[1].set_title(f"minimum at c = {cmin:.3f}   (sample mean {y.mean():.3f})\n"
                    f"loss here = {np.mean((y - c) ** 2):.3f}")
    plt.show()


w1 = dict(c=widgets.FloatSlider(value=0.0, min=-4, max=8, step=0.1,
                                description="prediction c:", **SL),
          outlier=widgets.FloatSlider(value=2.0, min=-2, max=14, step=0.5,
                                      description="one point at:", **SL))
display(widgets.HBox([w1["c"], w1["outlier"]]),
        widgets.interactive_output(draw_squares, w1))

Output()

## 2 — Absolute error and Huber: choosing what to ignore

Replace the square with the absolute value and the derivative stops growing:

$$\ell_{\text{abs}}=|y-\hat{y}|,\quad \frac{\partial\ell}{\partial\hat{y}}=-\text{sign}(y-\hat{y})\in\{-1,+1\}$$

Every point now pulls with the *same* force regardless of distance, so the balance point is where half the points sit on each side: **absolute error estimates the median**. An outlier ten units away and one just past the middle count equally, which is exactly the robustness you want and exactly the sensitivity you lose.

Huber splits the difference — quadratic near zero so it stays smooth and differentiable, linear beyond $\delta$ so influence is capped at $\delta$:

$$\ell_\delta(r)=\begin{cases}\tfrac12 r^2 & |r|\le\delta\\[2pt] \delta(|r|-\tfrac12\delta) & |r|>\delta\end{cases}
\qquad \frac{\partial\ell_\delta}{\partial r}=\text{clip}(r,-\delta,\delta)$$

Drag the outlier and watch the fitted slopes. With the true slope at 1.500, squared error degrades to 1.156, then 0.562, then **−0.344** — the line ends up pointing the wrong way because of one point. Absolute error stays at 1.500 throughout, and Huber holds at 1.406. (Slopes are quantised to the 0.031 spacing of the search grid.) The middle panel explains all of it: bounded derivative means bounded influence.

In [3]:
XL = np.linspace(0, 1, 20)
YL = 1.5 * XL + 0.3


def fit_line(x, y, kind, delta=0.5, n=161):
    """Dense 2-D search over (slope, intercept) — no optimizer needed."""
    S = np.linspace(-1.5, 3.5, n); B = np.linspace(-1.5, 2.5, n)
    R = y[None, None, :] - (S[:, None, None] * x[None, None, :] + B[None, :, None])
    L = {"squared": squared, "absolute": absolute,
         "huber": lambda r: huber(r, delta)}[kind](R).mean(-1)
    i, j = np.unravel_index(np.argmin(L), L.shape)
    return S[i], B[j]


def draw_robust(shift, delta):
    y = YL.copy(); y[3] += shift
    fits = {k: fit_line(XL, y, k, delta) for k in ("squared", "absolute", "huber")}

    fig, ax = plt.subplots(1, 3, figsize=(13.0, 3.8))
    fig.subplots_adjust(left=0.055, right=0.985, top=0.82, bottom=0.15, wspace=0.28)
    ax[0].plot(XL, y, "o", ms=6, color=CK, zorder=5)
    ax[0].plot(XL, 1.5 * XL + 0.3, color="0.6", lw=1.4, ls="--", label="true line")
    for (k, (s, b)), col in zip(fits.items(), (C1, C2, C3)):
        ax[0].plot(XL, s * XL + b, color=col, lw=2, label=f"{k}: slope {s:.3f}")
    ax[0].set_xlabel("x"); ax[0].set_ylabel("y"); ax[0].legend(fontsize=7.5)
    ax[0].set_title(f"one point moved by {shift:+.1f}")

    r = np.linspace(-3, 3, 400)
    ax[1].plot(r, squared(r), color=C1, lw=2, label="squared $r^2$")
    ax[1].plot(r, absolute(r), color=C2, lw=2, label="absolute $|r|$")
    ax[1].plot(r, huber(r, delta), color=C3, lw=2, label=f"Huber δ={delta:.1f}")
    ax[1].set_xlabel("residual r"); ax[1].set_ylabel("loss"); ax[1].set_ylim(0, 5)
    ax[1].legend(fontsize=7.5); ax[1].set_title("the losses")

    ax[2].plot(r, d_squared(r), color=C1, lw=2)
    ax[2].plot(r, d_absolute(r), color=C2, lw=2)
    ax[2].plot(r, d_huber(r, delta), color=C3, lw=2)
    ax[2].axhline(0, color="0.5", lw=0.8)
    ax[2].axhline(delta, color=C3, lw=0.8, ls=":")
    ax[2].axhline(-delta, color=C3, lw=0.8, ls=":")
    ax[2].set_xlabel("residual r"); ax[2].set_ylabel("$\\partial\\ell/\\partial r$")
    ax[2].set_ylim(-4, 4)
    ax[2].set_title("influence — squared error alone is unbounded")
    plt.show()


w2 = dict(shift=widgets.FloatSlider(value=0.0, min=0.0, max=10.0, step=0.5,
                                    description="move one point:", **SL),
          delta=widgets.FloatSlider(value=0.5, min=0.1, max=3.0, step=0.1,
                                    description="Huber δ:", **SL))
display(widgets.HBox([w2["shift"], w2["delta"]]),
        widgets.interactive_output(draw_robust, w2))

Output()

## 3 — Squared error on a classifier: the trap

Now predict a probability, $\hat{p}=\sigma(z)$, and keep squared error. The minimum is still in the right place, so nothing looks wrong — until you differentiate through the sigmoid:

$$\frac{\partial}{\partial z}\big(\sigma(z)-y\big)^2=2\big(\sigma(z)-y\big)\underbrace{\sigma(z)\big(1-\sigma(z)\big)}_{\to\,0\ \text{at both ends}}$$

That last factor is the problem. A positive example the model scores at $z=-8$ is as wrong as it could possibly be, and the gradient is $-6.7\times10^{-4}$ — the loss knows the answer is wrong and asks for almost no correction. Cross-entropy's gradient is simply $\sigma(z)-y=-1.000$, larger by a factor of **1491**.

The right panel measures what that costs. Starting logistic regression from a bad initialisation and counting steps to 95% accuracy:

| initial $w_0$ | squared error | cross-entropy | ratio |
|---|---|---|---|
| $-2$ | 16 | 3 | 5× |
| $-4$ | 109 | 6 | 18× |
| $-6$ | 524 | 8 | 66× |
| $-8$ | 2321 | 11 | 211× |
| $-9$ | 6461 | 13 | 497× |
| $-10$ | **never** (>8000) | 14 | — |

Cross-entropy grows linearly with how wrong you start — 3, 6, 8, 11, 13, 14 — while squared error roughly doubles per unit and runs off the end of the budget. That is $\sigma'(z)\sim e^{-|z|}$ showing up directly in the step count. This is the entire reason classifiers are not trained with squared error.

In [ ]:
def steps_to_target(loss, w0, lr=0.5, cap=8000, target=0.95, seed=0):
    rng = np.random.default_rng(seed)
    X = np.c_[np.r_[rng.normal(-1.5, .6, 150), rng.normal(1.5, .6, 150)], np.ones(300)]
    y = np.r_[np.zeros(150), np.ones(150)]
    w = np.array([w0, 0.0])
    for t in range(cap):
        p = sigmoid(X @ w)
        g = (X.T @ (2 * (p - y) * p * (1 - p)) if loss == "squared"
             else X.T @ (p - y)) / len(y)
        w = w - lr * g
        if ((sigmoid(X @ w) > 0.5) == (y == 1)).mean() >= target:
            return t
    return None


W0S = np.arange(-1.0, -10.5, -1.0)
STEPS = {k: [steps_to_target(k, w) for w in W0S] for k in ("squared", "cross-entropy")}


def draw_trap(z_mark):
    z = np.linspace(-10, 10, 600)
    p = sigmoid(z)
    g_sq = np.abs(2 * (p - 1) * p * (1 - p))
    g_ce = np.abs(p - 1)

    fig, ax = plt.subplots(1, 3, figsize=(13.0, 3.8))
    fig.subplots_adjust(left=0.055, right=0.985, top=0.82, bottom=0.15, wspace=0.3)
    ax[0].plot(z, (p - 1) ** 2, color=C1, lw=2, label="squared $(\\sigma(z)-1)^2$")
    ax[0].plot(z, -np.log(np.maximum(p, 1e-12)), color=C2, lw=2,
               label="cross-entropy $-\\log\\sigma(z)$")
    ax[0].axvline(z_mark, color=CK, lw=1.2, ls="--")
    ax[0].set_xlabel("logit z"); ax[0].set_ylabel("loss"); ax[0].set_ylim(0, 6)
    ax[0].legend(fontsize=7.5); ax[0].set_title("loss for a positive example (y = 1)")

    ax[1].semilogy(z, g_sq, color=C1, lw=2, label="squared")
    ax[1].semilogy(z, g_ce, color=C2, lw=2, label="cross-entropy")
    ax[1].axvline(z_mark, color=CK, lw=1.2, ls="--")
    pm = sigmoid(z_mark)
    gs = abs(2 * (pm - 1) * pm * (1 - pm)); gc = abs(pm - 1)
    ax[1].plot([z_mark, z_mark], [gs, gc], "o", color=CK, ms=6)
    ax[1].set_xlabel("logit z"); ax[1].set_ylabel("$|\\partial\\ell/\\partial z|$")
    ax[1].set_ylim(1e-6, 3); ax[1].legend(fontsize=7.5)
    ax[1].set_title(f"at z = {z_mark:+.1f}: squared {gs:.2e}, "
                    f"CE {gc:.3f}\ncross-entropy pulls "
                    f"{gc / max(gs, 1e-300):.0f}× harder")

    sq = [s if s is not None else np.nan for s in STEPS["squared"]]
    ce = [s if s is not None else np.nan for s in STEPS["cross-entropy"]]
    ax[2].semilogy(-W0S, sq, "o-", color=C1, lw=2, ms=5, label="squared")
    ax[2].semilogy(-W0S, ce, "o-", color=C2, lw=2, ms=5, label="cross-entropy")
    ax[2].set_xlabel("how wrong the start is  $|w_0|$")
    ax[2].set_ylabel("steps to 95% accuracy")
    ax[2].legend(fontsize=7.5)
    ax[2].set_title("exponential vs linear\n(missing points = did not converge)")
    plt.show()


w3 = dict(z_mark=widgets.FloatSlider(value=-8.0, min=-10, max=10, step=0.5,
                                     description="logit z:", **SL))
display(w3["z_mark"], widgets.interactive_output(draw_trap, w3))

FloatSlider(value=-8.0, continuous_update=False, description='logit z:', layout=Layout(width='330px'), max=10.…

Output()

## 4 — Binary cross-entropy, derived rather than invented

Treat the label as a Bernoulli draw with parameter $\hat{p}$. The likelihood of observing $y$ is $\hat{p}^{\,y}(1-\hat{p})^{1-y}$, and the negative log-likelihood of the dataset is

$$\ell(y,\hat{p})=-\big[y\log\hat{p}+(1-y)\log(1-\hat{p})\big]$$

Nothing was designed here; the loss is what maximum likelihood hands you. Its two useful properties both fall out immediately.

It is **unbounded**: as $\hat{p}\to0$ for a positive example the penalty goes to $+\infty$. Confident and wrong is punished without limit, which is why a single badly-predicted example can dominate a batch, and why probabilities get clipped in practice.

And composed with the sigmoid the messy factors cancel exactly:

$$\frac{\partial\ell}{\partial z}=\underbrace{\left(\frac{\hat{p}-y}{\hat{p}(1-\hat{p})}\right)}_{\partial\ell/\partial\hat p}\cdot\underbrace{\hat{p}(1-\hat{p})}_{\sigma'(z)}=\hat{p}-y$$

The gradient is the *prediction error itself* — no saturating factor left. That cancellation is the whole reason this particular pairing is standard. The third panel checks it by differentiating the loss numerically and overlaying $\hat p-y$; the residual $\sim10^{-5}$ is the finite-difference error of the check, not a gap in the identity, which is exact.

In [ ]:
def draw_bce(p_mark, y_true):
    p = np.linspace(1e-4, 1 - 1e-4, 800)
    loss = -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

    fig, ax = plt.subplots(1, 3, figsize=(13.0, 3.8))
    fig.subplots_adjust(left=0.055, right=0.985, top=0.82, bottom=0.15, wspace=0.3)
    ax[0].plot(p, -np.log(p), color=C2, lw=2, label="y = 1:  $-\\log\\hat{p}$")
    ax[0].plot(p, -np.log(1 - p), color=C1, lw=2,
               label="y = 0:  $-\\log(1-\\hat{p})$")
    ax[0].axvline(p_mark, color=CK, lw=1.2, ls="--")
    cur = -(y_true * np.log(p_mark) + (1 - y_true) * np.log(1 - p_mark))
    ax[0].plot([p_mark], [cur], "o", color=CK, ms=7, zorder=5)
    ax[0].set_xlabel("predicted probability $\\hat{p}$"); ax[0].set_ylabel("loss")
    ax[0].set_ylim(0, 6); ax[0].legend(fontsize=7.5)
    ax[0].set_title(f"true label y = {y_true} → loss {cur:.3f}"
                    + ("   (unbounded as $\\hat p\\to0$)" if y_true == 1 else ""))

    ax[1].plot(p, loss, color=C3, lw=2)
    ax[1].fill_between(p, 0, loss, color=C3, alpha=0.15)
    ax[1].axvline(p_mark, color=CK, lw=1.2, ls="--")
    ax[1].axvline(y_true, color=C4, lw=1.6, ls=":", label="minimum at $\\hat p = y$")
    ax[1].set_xlabel("$\\hat{p}$"); ax[1].set_ylabel("loss"); ax[1].set_ylim(0, 6)
    ax[1].legend(fontsize=7.5)
    ax[1].set_title("the loss is minimised exactly at the truth")

    z = np.linspace(-8, 8, 500); pz = sigmoid(z)
    ax[2].plot(z, pz - y_true, color=C2, lw=2.4, label="$\\hat p - y$  (analytic)")
    num = np.gradient(-(y_true * np.log(np.maximum(pz, 1e-12))
                        + (1 - y_true) * np.log(np.maximum(1 - pz, 1e-12))), z)
    ax[2].plot(z[::12], num[::12], "o", color=CK, ms=4, label="numerical $d\\ell/dz$")
    ax[2].axhline(0, color="0.5", lw=0.8)
    ax[2].set_xlabel("logit z"); ax[2].set_ylabel("$\\partial\\ell/\\partial z$")
    ax[2].set_ylim(-1.15, 1.15); ax[2].legend(fontsize=7.5)
    ax[2].set_title(f"max discrepancy {np.max(np.abs(num - (pz - y_true))):.2e}"
                    "\nthe sigmoid factor cancels exactly")
    plt.show()


w4 = dict(p_mark=widgets.FloatSlider(value=0.15, min=0.01, max=0.99, step=0.01,
                                     description="p_hat:", **SL),
          y_true=widgets.Dropdown(options=[1, 0], value=1, description="true y:",
                                  style={"description_width": "108px"},
                                  layout=widgets.Layout(width="300px")))
display(widgets.HBox([w4["p_mark"], w4["y_true"]]),
        widgets.interactive_output(draw_bce, w4))

Output()

## 5 — More than two classes: softmax and the simplex

With $K$ classes the model emits a score per class and softmax turns them into a distribution; the loss is the same negative log-likelihood, now picking out one coordinate:

$$\hat{p}_k=\frac{e^{z_k/T}}{\sum_j e^{z_j/T}},\qquad
\ell=-\sum_k q_k\log\hat{p}_k=-\log\hat{p}_{\text{true}}$$

For a one-hot target the three standard descriptions coincide exactly — verified numerically in the panel title:

$$\underbrace{H(q,\hat{p})}_{\text{cross-entropy}}=\underbrace{-\log\hat{p}_{\text{true}}}_{\text{neg. log-likelihood}}=\underbrace{D_{\text{KL}}(q\,\|\,\hat{p})+H(q)}_{\text{KL divergence},\ H(q)=0}$$

Every valid distribution over 3 classes is a point in a triangle, and the loss is a field over it: zero at the true corner, infinite along the opposite edge. Only the *differences* between logits matter — adding a constant to all of them changes nothing, so softmax has one redundant degree of freedom.

Temperature $T$ rescales those differences. $T\to0$ drives the point to a corner (a hard argmax, confident and uncalibrated); $T\to\infty$ drives it to the centre (uniform, maximum entropy). It is the knob behind both distillation and sampling temperature in language models.

In [6]:
def tri_xy(p):
    """Barycentric coordinates to 2-D, for plotting a 3-class simplex."""
    V = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3) / 2]])
    return np.asarray(p) @ V


def draw_simplex(z1, z2, z3, T, true_k):
    z = np.array([z1, z2, z3], dtype=float)
    p = softmax(z / T)
    q = np.eye(3)[true_k]
    ce = float(-np.sum(q * np.log(np.maximum(p, 1e-12))))
    nll = float(-np.log(max(p[true_k], 1e-12)))
    kl = float(np.sum(q[q > 0] * np.log(q[q > 0] / np.maximum(p[q > 0], 1e-12))))

    fig, ax = plt.subplots(1, 3, figsize=(13.0, 3.9))
    fig.subplots_adjust(left=0.04, right=0.985, top=0.80, bottom=0.1, wspace=0.26)

    n = 220
    a, b = np.meshgrid(np.linspace(0, 1, n), np.linspace(0, 1, n))
    c = 1 - a - b
    ok = (c >= 0)
    P = np.stack([a, b, c], -1)
    L = np.where(ok, -np.log(np.maximum(P[..., true_k], 1e-6)), np.nan)
    XY = tri_xy(P.reshape(-1, 3)).reshape(n, n, 2)
    ax[0].contourf(XY[..., 0], XY[..., 1], np.clip(L, 0, 6), levels=24, cmap="magma")
    V = tri_xy(np.eye(3))
    ax[0].plot(*np.r_[V, V[:1]].T, color="k", lw=1.4)
    for i, lab in enumerate(["class 0", "class 1", "class 2"]):
        ax[0].annotate(lab, tri_xy(np.eye(3)[i]), fontsize=8.5,
                       ha="center", va="center",
                       xytext=(0, 12 if i == 2 else -12),
                       textcoords="offset points",
                       color=(C4 if i == true_k else "0.3"))
    ax[0].plot(*tri_xy(p), "o", ms=11, color=C4, mec="w", mew=1.5, zorder=5)
    ax[0].set_aspect("equal"); ax[0].axis("off")
    ax[0].set_title(f"loss field over the simplex\ntrue class = {true_k}")

    ax[1].bar(np.arange(3) - 0.2, softmax(z), width=0.36, color="0.7",
              label="T = 1")
    ax[1].bar(np.arange(3) + 0.2, p, width=0.36, color=C3, label=f"T = {T:.2f}")
    ax[1].set_xticks(range(3)); ax[1].set_xticklabels(["class 0", "class 1", "class 2"])
    ax[1].set_ylim(0, 1.05); ax[1].legend(fontsize=7.5)
    ax[1].set_title(f"H(q,p) = {ce:.4f}   −log p_true = {nll:.4f}\n"
                    f"KL(q‖p) + H(q) = {kl:.4f} + 0")

    Ts = np.logspace(-1, 1.4, 200)
    curves = np.array([softmax(z / t) for t in Ts])
    for k, col in enumerate((C1, C2, C4)):
        ax[2].semilogx(Ts, curves[:, k], color=col, lw=2, label=f"class {k}")
    ax[2].semilogx(Ts, [-np.log(max(softmax(z / t)[true_k], 1e-12)) / 6 for t in Ts],
                   color=CK, lw=1.4, ls="--", label="loss (scaled)")
    ax[2].axvline(T, color=CK, lw=1.2, ls=":")
    ax[2].set_xlabel("temperature T"); ax[2].set_ylabel("probability")
    ax[2].set_ylim(0, 1.05); ax[2].legend(fontsize=7)
    ax[2].set_title("T → 0 gives a hard argmax, T → ∞ gives uniform")
    plt.show()


w5 = dict(z1=widgets.FloatSlider(value=2.0, min=-4, max=6, step=0.25,
                                 description="logit z_0:", **SL),
          z2=widgets.FloatSlider(value=1.0, min=-4, max=6, step=0.25,
                                 description="logit z_1:", **SL),
          z3=widgets.FloatSlider(value=-0.5, min=-4, max=6, step=0.25,
                                 description="logit z_2:", **SL),
          T=widgets.FloatSlider(value=1.0, min=0.1, max=8.0, step=0.1,
                                description="temperature T:", **SL),
          true_k=widgets.Dropdown(options=[0, 1, 2], value=0,
                                  description="true class:",
                                  style={"description_width": "108px"},
                                  layout=widgets.Layout(width="300px")))
display(widgets.VBox([widgets.HBox([w5["z1"], w5["z2"], w5["z3"]]),
                      widgets.HBox([w5["T"], w5["true_k"]])]),
        widgets.interactive_output(draw_simplex, w5))

Output()

## 6 — One picture that contains all of them

Every binary classification loss is a function of a single number, the **margin** $m=y\cdot f(x)$ with $y\in\{-1,+1\}$. Positive margin means correct, and larger means more confident. What we would actually like to minimise is the count of mistakes,

$$\ell_{0\text{-}1}(m)=\mathbb{1}[m\le0]$$

which is flat everywhere and discontinuous at zero — zero gradient almost everywhere, so gradient descent cannot touch it. Every practical loss is a differentiable **upper bound** on it, and the cell verifies the bound holds at every sampled margin:

| loss | $\ell(m)$ | behaviour |
|---|---|---|
| hinge (SVM) | $\max(0,1-m)$ | gradient exactly $0$ past $m=1$ — only support vectors matter |
| logistic | $\log_2(1+e^{-m})$ | never exactly zero: every point keeps nudging forever |
| exponential (AdaBoost) | $e^{-m}$ | grows without bound — extremely outlier-sensitive |
| squared | $(1-m)^2$ | penalises being *too* correct, which is why it misclassifies |

The base-2 logarithm is not decoration: $\log_2(1+e^{0})=1$ exactly, so the logistic curve passes through the corner of the 0-1 step. With natural logs it would be $0.693<1$ and would fail to bound it at $m=0$. In fact all four surrogates pass through exactly $1$ at $m=0$ — they are all calibrated to touch the step at the decision boundary and can only differ in how they leave it.

The right panel is the one to remember. Squared error is the only curve that **turns back upward** for large positive margins — it actively penalises confident correct predictions, which is a second, independent reason not to use it for classification.

In [7]:
LOSSES = {
    "hinge  max(0, 1−m)": (lambda m: np.maximum(0, 1 - m),
                           lambda m: np.where(m < 1, -1.0, 0.0), C1),
    "logistic  log₂(1+e⁻ᵐ)": (lambda m: np.log2(1 + np.exp(-np.clip(m, -50, 50))),
                              lambda m: -1 / (np.log(2) * (1 + np.exp(np.clip(m, -50, 50)))), C2),
    "exponential  e⁻ᵐ": (lambda m: np.exp(-np.clip(m, -50, 50)),
                         lambda m: -np.exp(-np.clip(m, -50, 50)), C3),
    "squared  (1−m)²": (lambda m: (1 - m) ** 2, lambda m: -2 * (1 - m), C4),
}


def draw_margin(shown, mmark):
    m = np.linspace(-3, 3, 700)
    fig, ax = plt.subplots(1, 2, figsize=(11.6, 4.0))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.82, bottom=0.14, wspace=0.24)

    ax[0].step(m, (m <= 0).astype(float), where="post", color=CK, lw=2.4,
               label="0-1 loss (what we want)")
    bound_ok = True
    for name, (fn, dfn, col) in LOSSES.items():
        if name.split()[0] not in shown:
            continue
        v = fn(m)
        ax[0].plot(m, v, color=col, lw=2, label=name)
        bound_ok &= bool(np.all(v >= (m <= 0).astype(float) - 1e-9))
        ax[1].plot(m, dfn(m), color=col, lw=2, label=name)
    ax[0].axvline(mmark, color="0.4", lw=1.2, ls="--")
    ax[0].set_xlabel("margin  m = y·f(x)"); ax[0].set_ylabel("loss")
    ax[0].set_ylim(-0.2, 5); ax[0].legend(fontsize=7.5)
    ax[0].set_title(f"all shown curves upper-bound the 0-1 loss: {bound_ok}")

    ax[1].axvline(mmark, color="0.4", lw=1.2, ls="--")
    ax[1].axhline(0, color="0.5", lw=0.8)
    ax[1].axvspan(1, 3, color="0.9", zorder=0)
    ax[1].text(2.0, 1.6, "correct with\nmargin > 1", fontsize=7.5, ha="center",
               color="0.35")
    ax[1].set_xlabel("margin m"); ax[1].set_ylabel("$\\partial\\ell/\\partial m$")
    ax[1].set_ylim(-4, 4); ax[1].legend(fontsize=7.5)
    rows = [f"{n.split()[0]}: {LOSSES[n][1](np.array([mmark]))[0]:+.3f}"
            for n in LOSSES if n.split()[0] in shown]
    ax[1].set_title(f"gradient at m = {mmark:+.2f}   |   " + ",  ".join(rows))
    plt.show()


w6 = dict(shown=widgets.SelectMultiple(
              options=["hinge", "logistic", "exponential", "squared"],
              value=("hinge", "logistic", "exponential", "squared"),
              description="show:", rows=4,
              style={"description_width": "60px"},
              layout=widgets.Layout(width="260px")),
          mmark=widgets.FloatSlider(value=2.0, min=-3, max=3, step=0.1,
                                    description="margin m:", **SL))
display(widgets.HBox([w6["shown"], w6["mmark"]]),
        widgets.interactive_output(draw_margin, w6))

Output()